# Lenguajes y Autómatas II (SCD-1016)
## Sesión 14 · Declaración vs Uso de Variables

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo detecta el compilador que usaste una variable que nunca declaraste?
>
> **Objetivo de la sesión:** Implementar las dos operaciones que sostienen la regla de declaración previa: insertar en el ámbito actual y resolver un nombre declarado, distinguiendo uso válido, uso no declarado y redeclaración.

**Entregable:** tus funciones declarar() y usar() en artesano_sesion14_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion14.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes una tabla vacía y una secuencia de operaciones; declarar() añade un nombre solo si no existe en ese mismo ámbito y usar() responde si ya fue declarado. La diferencia entre "no existe" y "existe al lado" es lo que mide el harness.

Al final de la sesión tendrás: **Una tabla de símbolos mínima que acepta declaraciones nuevas, rechaza repetidos y responde si un nombre puede usarse.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 14](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion14.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion14.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion14.h"

int declarar(Tabla *tabla, const char *nombre);
int usar(const Tabla *tabla, const char *nombre);
```

- declarar() devuelve 1 si insertó, 0 si el nombre ya estaba y -1 si recibe NULL o un nombre vacío.
- usar() devuelve 1 solo si el nombre ya fue declarado; en cualquier otro caso devuelve 0.
- La comparación es exacta: "x" y "xy" son nombres distintos.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion14_c.c` junto con `artesano_sesion14_c.c`
> (o `ia_sesion14_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `usar sin declarar` | 0 |
| 2 | `usar despues de declarar` | 1 |
| 3 | `redeclarar el mismo nombre` | 0 |
| 4 | `total tras redeclarar` | 1 |
| 5 | `prefijo no es el mismo nombre` | 0 |
| 6 | `declarar nombre vacio` | -1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 14 - Declaracion previa y redeclaracion ===
Caso                               esperado:              obtenido:              resultado
[ 1] usar sin declarar                  esperado: 0            obtenido: 0            OK
[ 2] usar despues de declarar           esperado: 1            obtenido: 1            OK
[ 3] redeclarar el mismo nombre         esperado: 0            obtenido: 0            OK
[ 4] total tras redeclarar              esperado: 1            obtenido: 1            OK
[ 5] prefijo no es el mismo nombre      esperado: 0            obtenido: 0            OK
[ 6] declarar nombre vacio              esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion14.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] usar() distingue un nombre declarado de uno inexistente sin depender de comparaciones parciales.
- [ ] declarar() rechaza el duplicado y deja el total de símbolos intacto.
- [ ] El caso de nombre vacío devuelve -1 sin insertar nada.


## Paso 0 · Entorno


In [2]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion14_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [3]:
%%writefile api_sesion14.h
/* api_sesion14.h - contrato de la Sesion 14
 * Declaración previa y redeclaración
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION14_H
#define API_SESION14_H

#define TABLA_MAX 16
typedef struct Tabla {
    const char *nombres[TABLA_MAX];
    int total;
} Tabla;

int declarar(Tabla *tabla, const char *nombre);
int usar(const Tabla *tabla, const char *nombre);

#endif /* API_SESION14_H */


Writing api_sesion14.h


In [4]:
%%writefile artesano_sesion14_c.c
/* artesano_sesion14_c.c - Fase 1: El Artesano (sin IA)
 * Declaración previa y redeclaración.
 *
 * Implementa lo que declara api_sesion14.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */

#include "api_sesion14.h"
#include <stdio.h>
#include <string.h>

int declarar(Tabla *tabla, const char *nombre) {

    if (tabla == NULL || nombre == NULL || nombre[0] == '\0') {
        return -1;
    }

    for (int i = 0; i < tabla->total; i++) {
        if (strcmp(tabla->nombres[i], nombre) == 0) {
            return 0;
        }
    }

    if (tabla->total >= TABLA_MAX) {
        return -1;
    }

    tabla->nombres[tabla->total] = nombre;
    tabla->total++;

    return 1;
}

int usar(const Tabla *tabla, const char *nombre) {

    if (tabla == NULL || nombre == NULL || nombre[0] == '\0') {
        return 0;
    }

    for (int i = 0; i < tabla->total; i++) {
        if (strcmp(tabla->nombres[i], nombre) == 0) {
            return 1;
        }
    }

    return 0;
}


Writing artesano_sesion14_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion14_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [5]:
%%writefile ia_sesion14_c.c
/* ia_sesion14_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion14.h. No puede traer main().
 */
#include "api_sesion14.h"

/* TODO: pega aqui la propuesta de la IA */

#include "api_sesion14.h"
#include <string.h>

int declarar(Tabla *tabla, const char *nombre) {

    if (tabla == NULL || nombre == NULL || nombre[0] == '\0') {
        return -1;
    }

    for (int i = 0; i < tabla->total; i++) {
        if (strcmp(tabla->nombres[i], nombre) == 0) {
            return 0;
        }
    }

    if (tabla->total >= TABLA_MAX) {
        return -1;
    }

    tabla->nombres[tabla->total] = nombre;
    tabla->total++;

    return 1;
}

int usar(const Tabla *tabla, const char *nombre) {

    if (tabla == NULL || nombre == NULL || nombre[0] == '\0') {
        return 0;
    }

    for (int i = 0; i < tabla->total; i++) {
        if (strcmp(tabla->nombres[i], nombre) == 0) {
            return 1;
        }
    }

    return 0;
}


Writing ia_sesion14_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [6]:
%%writefile harness_sesion14_c.c
/* harness_sesion14_c.c - auditoria automatica de la Sesion 14
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion14_c.c artesano_sesion14_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion14.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 14 - Declaracion previa y redeclaracion ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Tabla t; memset(&t, 0, sizeof t); int r = usar(&t, "x"); verificar_int("usar sin declarar", (long)(0), (long)(r)); }
    { Tabla t; memset(&t, 0, sizeof t); declarar(&t, "x"); int r = usar(&t, "x"); verificar_int("usar despues de declarar", (long)(1), (long)(r)); }
    { Tabla t; memset(&t, 0, sizeof t); declarar(&t, "x"); int r = declarar(&t, "x"); verificar_int("redeclarar el mismo nombre", (long)(0), (long)(r)); }
    { Tabla t; memset(&t, 0, sizeof t); declarar(&t, "x"); declarar(&t, "x"); int r = t.total; verificar_int("total tras redeclarar", (long)(1), (long)(r)); }
    { Tabla t; memset(&t, 0, sizeof t); declarar(&t, "x"); int r = usar(&t, "xy"); verificar_int("prefijo no es el mismo nombre", (long)(0), (long)(r)); }
    { Tabla t; memset(&t, 0, sizeof t); int r = declarar(&t, ""); verificar_int("declarar nombre vacio", (long)(-1), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion14.md" "\n");
    return 0;
}


Writing harness_sesion14_c.c


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion14_c.c artesano_sesion14_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 14 - Declaracion previa y redeclaracion ===
Caso                               esperado:              obtenido:              resultado
[ 1] usar sin declarar                  esperado: 0            obtenido: 0            OK
[ 2] usar despues de declarar           esperado: 1            obtenido: 1            OK
[ 3] redeclarar el mismo nombre         esperado: 0            obtenido: 0            OK
[ 4] total tras redeclarar              esperado: 1            obtenido: 1            OK
[ 5] prefijo no es el mismo nombre      esperado: 0            obtenido: 0            OK
[ 6] declarar nombre vacio              esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion14.md


In [8]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion14_c.c ia_sesion14_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 14 - Declaracion previa y redeclaracion ===
Caso                               esperado:              obtenido:              resultado
[ 1] usar sin declarar                  esperado: 0            obtenido: 0            OK
[ 2] usar despues de declarar           esperado: 1            obtenido: 1            OK
[ 3] redeclarar el mismo nombre         esperado: 0            obtenido: 0            OK
[ 4] total tras redeclarar              esperado: 1            obtenido: 1            OK
[ 5] prefijo no es el mismo nombre      esperado: 0            obtenido: 0            OK
[ 6] declarar nombre vacio              esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion14.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion14.md` a `auditorias/auditoria_sesion14.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
